# TODO 6.3：GPTQ INT4 实验

使用与 AWQ 相同的校准集、bit 数和 group size 32/64/128。完整模型写入 `banking_llm_project/gptq`，分析文件写入 `OptimizationSystem/out/gptq`。

首次运行先执行安装单元，然后通过 `运行时 → 重启会话` 重启。重启后跳过安装单元，从挂载 Drive 单元继续。

In [5]:
%pip -q install -U gptqmodel==7.5.0 optimum accelerate safetensors
%pip -q install -U transformers==5.18.0
print('安装完成。现在必须重启 Colab 会话，不要继续运行下一个单元。')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 133.7 MB/s eta 0:00:00
安装完成。现在必须重启 Colab 会话，不要继续运行下一个单元。


In [1]:
from google.colab import drive
drive.mount('/content/drive')

import importlib.metadata
import json
import sys
from pathlib import Path
import torch

print('transformers:', importlib.metadata.version('transformers'))
print('gptqmodel:', importlib.metadata.version('gptqmodel'))
from transformers import AutoModelForCausalLM, AutoTokenizer, GPTQConfig
print('Transformers GPTQ 依赖导入：PASS')

PROJECT_DIR = Path('/content/drive/MyDrive/OptimizationSystem')
DATASET_DIR = Path('/content/drive/MyDrive/banking_llm_project/datasets')
MODEL_MANIFEST = Path('/content/drive/MyDrive/banking_llm_project/model_input_real/model_manifest.json')
CONFIG_PATH = PROJECT_DIR / 'configs/gptq.json'
CALIBRATION_MANIFEST = PROJECT_DIR / 'out/dataset_input/dataset_manifest.json'
RESULT_DIR = PROJECT_DIR / 'out/gptq'
MODEL_RESULT_DIR = Path('/content/drive/MyDrive/banking_llm_project/gptq')
sys.path.insert(0, str(PROJECT_DIR))

from src.data.finqa_assets import load_assets
from src.quantization.gptq import (
    build_gptq_plans, collect_gptq_evidence,
    run_gptq_plan, validate_gptq_evidence,
)

assets = load_assets(DATASET_DIR)
config = json.loads(CONFIG_PATH.read_text(encoding='utf-8'))
plans = build_gptq_plans(config)
assert torch.cuda.is_available(), '请切换到 NVIDIA GPU。'
print(torch.cuda.get_device_name(0), [plan.plan_id for plan in plans])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
transformers: 5.18.0
gptqmodel: 7.5.0
Transformers GPTQ 依赖导入：PASS
NVIDIA A100-SXM4-40GB ['gptq-int4-w4a16-g32', 'gptq-int4-w4a16-g64', 'gptq-int4-w4a16-g128']


In [2]:
import gc

results = []
for plan in plans:
    manifest = RESULT_DIR / plan.plan_id / 'quantized_model_manifest.json'
    if manifest.is_file():
        print('跳过已完成:', plan.plan_id)
        continue
    result = run_gptq_plan(
        plan=plan,
        model_manifest_file=MODEL_MANIFEST,
        calibration_prompts=assets.calibration_prompts,
        output_dir=RESULT_DIR / plan.plan_id,
        model_output_dir=MODEL_RESULT_DIR / plan.plan_id,
        max_calibration_samples=int(config['max_calibration_samples']),
        max_calibration_seq_len=int(config['max_calibration_seq_len']),
    )
    results.append(result)
    if result['status'] != 'pass':
        raise RuntimeError(result['detail'])
    gc.collect()
    torch.cuda.empty_cache()
print(json.dumps(results, ensure_ascii=False, indent=2))

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!



INFO  LogBar: headless/CI mode; progress animations disabled.                   
WARN  Python GIL is enabled: Multi-gpu quant acceleration for MoE models is sub-optimal and multi-core accelerated cpu packing is also disabled. We recommend Python >= 3.13.3t with Pytorch > 2.8 for mult-gpu quantization and multi-cpu packing with env `PYTHON_GIL=0`.
INFO  ENV: Auto setting PYTORCH_ALLOC_CONF='expandable_segments:True,max_split_size_mb:256,garbage_collection_threshold:0.7' for memory saving.
INFO  ENV: Auto setting CUDA_DEVICE_ORDER=PCI_BUS_ID for correctness.           
INFO  

┌─────────────┐    ┌────────────────────────┐    ┌────────────┐    ┌─────────┐
│ GPT-QModel  │ -> │ ▓▓▓▓▓▓▓▓▓▓▓▓ 16bit     │ -> │ ▒▒▒▒ 8bit  │ -> │ ░░ 4bit │
└─────────────┘    └────────────────────────┘    └────────────┘    └─────────┘
GPT-QModel   : 7.5.0
Transformers : 5.18.0
Torch        : 2.11.0+cu130
Triton       : 3.6.0


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Quantizing model.layers blocks :   0%|          | 0/24 [00:00<?, ?it/s]

Quantizing layers inside the block:   0%|          | 0/7 [00:00<?, ?it/s]

...

Quantizing layers inside the block:   0%|          | 0/7 [00:00<?, ?it/s]

Quantizing layers inside the block:   0%|          | 0/7 [00:00<?, ?it/s]

INFO  Packing Kernel: Auto-selection: adding candidate `TritonV2Linear`         
INFO  Kernel: selected -> `TritonV2Linear`.                                     
INFO  gc.collect() reclaimed 317 objects in 0.333s                              


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[
  {
    "plan": {
      "plan_id": "gptq-int4-w4a16-g32",
      "method": "gptq",
      "bits": 4,
      "weight_bits": 4,
      "activation_bits": 16,
      "group_size": 32,
      "quantization_granularity": "groupwise",
      "backend_package": "gptqmodel"
    },
    "status": "pass",
    "output_dir": "/content/drive/MyDrive/OptimizationSystem/out/gptq/gptq-int4-w4a16-g32",
    "model_output_dir": "/content/drive/MyDrive/banking_llm_project/gptq/gptq-int4-w4a16-g32"
  },
  {
    "plan": {
      "plan_id": "gptq-int4-w4a16-g64",
      "method": "gptq",
      "bits": 4,
      "weight_bits": 4,
      "activation_bits": 16,
      "group_size": 64,
      "quantization_granularity": "groupwise",
      "backend_package": "gptqmodel"
    },
    "status": "pass",
    "output_dir": "/content/drive/MyDrive/OptimizationSystem/out/gptq/gptq-int4-w4a16-g64",
    "model_output_dir": "/content/drive/MyDrive/banking_llm_project/gptq/gptq-int4-w4a16-g64"
  },
  {
    "plan": {
      "plan_id": "gp

In [3]:
evidence = collect_gptq_evidence(
    CONFIG_PATH, CALIBRATION_MANIFEST, MODEL_MANIFEST, RESULT_DIR
)
evidence_path = RESULT_DIR / 'gptq_evidence.json'
evidence_path.write_text(
    json.dumps(evidence, ensure_ascii=False, indent=2) + '\n',
    encoding='utf-8',
)
report = validate_gptq_evidence(evidence_path)
(RESULT_DIR / 'validation.json').write_text(
    json.dumps(report.to_dict(), ensure_ascii=False, indent=2) + '\n',
    encoding='utf-8',
)
print(json.dumps(report.to_dict(), ensure_ascii=False, indent=2))
assert report.complete, 'GPTQ 三个 group size 未全部完成。'
print('TODO 6.3 GPTQ 导出验收：PASS')

{
  "evidence": "/content/drive/MyDrive/OptimizationSystem/out/gptq/gptq_evidence.json",
  "complete": true,
  "checks": [
    {
      "name": "gptq_awq_comparable_matrix",
      "status": "pass",
      "detail": "GPTQ uses INT4 group sizes 32, 64, and 128."
    },
    {
      "name": "gptq_backend_recorded",
      "status": "pass",
      "detail": "GPTQModel version is recorded."
    },
    {
      "name": "gptq_quantized_artifacts",
      "status": "pass",
      "detail": "Each GPTQ plan has config, manifest, and event log."
    },
    {
      "name": "gptq_real_execution",
      "status": "pass",
      "detail": "All GPTQ plans completed real export."
    }
  ]
}
TODO 6.3 GPTQ 导出验收：PASS


In [4]:
import gc, json, sys
from pathlib import Path
import torch
PROJECT_DIR=Path('/content/drive/MyDrive/OptimizationSystem')
DATASET_DIR=Path('/content/drive/MyDrive/banking_llm_project/datasets')
MODEL_MANIFEST=Path('/content/drive/MyDrive/banking_llm_project/model_input_real/model_manifest.json')
BASELINE_CONFIG=PROJECT_DIR/'configs/baseline.json'
QUALITY_CONFIG=PROJECT_DIR/'configs/baseline_quality.json'
BENCHMARK_CONFIG=PROJECT_DIR/'configs/quantization_benchmark.json'
GPTQ_DIR=PROJECT_DIR/'out/gptq'
AWQ_DIR=PROJECT_DIR/'out/awq'
sys.path.insert(0,str(PROJECT_DIR))
from src.baseline.service import BaselineService, InferenceRequest
from src.data.finqa_assets import load_assets
from src.evaluation.baseline_quality import run_baseline_quality, validate_quality_report
from src.evaluation.finqa_metrics import classify_numeric_regression
from src.benchmark.runner import run_benchmark, validate_benchmark_report
from src.quantization.gptq import load_gptq_service
from src.quantization.awq import load_awq_service
assets=load_assets(DATASET_DIR)
quality_config=json.loads(QUALITY_CONFIG.read_text(encoding='utf-8'))
benchmark_config=json.loads(BENCHMARK_CONFIG.read_text(encoding='utf-8'))
print(torch.cuda.get_device_name(0),len(assets.quality_dev),len(assets.high_risk_regression))

NVIDIA A100-SXM4-40GB 883 91


In [5]:
def predict_rows(service,dataset,prefix):
    rows=[]; batch_size=int(quality_config['batch_size'])
    for start in range(0,len(dataset),batch_size):
        batch=[dataset[index] for index in range(start,min(start+batch_size,len(dataset)))]
        requests=[InferenceRequest.from_finqa_row(row,request_id=f'{prefix}-{start+offset+1:06d}') for offset,row in enumerate(batch)]
        results=service.generate_batch(requests,prompt_suffix=quality_config.get('prompt_suffix',''))
        for row,request,result in zip(batch,requests,results):
            rows.append({'question':request.question,'prediction':result['output_text'],'reference':row.get('answer') or row.get('qa',{}).get('answer')})
    return rows
baseline=BaselineService.from_local_model(BASELINE_CONFIG,PROJECT_DIR/'out/model_input_real/model_manifest.json',GPTQ_DIR/'fp16_regression_service.jsonl')
baseline_regression=predict_rows(baseline,assets.high_risk_regression,'fp16-regression')
del baseline; gc.collect(); torch.cuda.empty_cache()

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [6]:
for group_size in (32,64,128):
    plan_id=f'gptq-int4-w4a16-g{group_size}'; plan_dir=GPTQ_DIR/plan_id
    service=load_gptq_service(BASELINE_CONFIG,plan_dir/'quantized_model_manifest.json',MODEL_MANIFEST,plan_dir/'gptq_quality_service.jsonl')
    quality_dir=plan_dir/'quality'
    quality=run_baseline_quality(service,assets.quality_dev,assets.manifest,quality_config,quality_dir)
    quality_file=quality_dir/'baseline_quality_report.json'
    quality_file.write_text(json.dumps(quality,ensure_ascii=False,indent=2)+'\n',encoding='utf-8')
    validation=validate_quality_report(quality_file)
    (quality_dir/'validation.json').write_text(json.dumps(validation.to_dict(),ensure_ascii=False,indent=2)+'\n',encoding='utf-8')
    candidate=predict_rows(service,assets.high_risk_regression,f'{plan_id}-regression')
    baseline_map={row['question']:row for row in baseline_regression}
    comparisons=[{**classify_numeric_regression(baseline_map[row['question']]['prediction'],row['prediction'],row['reference'],float(quality_config.get('tolerance',1e-4))),'question':row['question']} for row in candidate]
    regression={'schema_version':1,'plan_id':plan_id,'dataset_role':'high_risk_regression','total':len(comparisons),'new_regressions':sum(row['new_regression'] for row in comparisons),'critical_errors':sum(row['critical_error_type'] is not None for row in comparisons),'rows':comparisons}
    (plan_dir/'regression.json').write_text(json.dumps(regression,ensure_ascii=False,indent=2)+'\n',encoding='utf-8')
    print(plan_id,quality['metrics'],regression['new_regressions'])
    del service; gc.collect(); torch.cuda.empty_cache()

INFO  Kernel: Auto-selection: adding candidate `MarlinLinear`                   
INFO  Kernel: selected -> `MarlinLinear`.                                       


Loading weights:   0%|          | 0/794 [00:00<?, ?it/s]

INFO  Marlin fp16: compiling torch.ops JIT extension in `/root/.cache/gptqmodel/torch_extensions/marlin_fp16/3380075ecccdabdf`.
INFO  Marlin fp16: torch.ops JIT extension ready in 184s (estimated ~117s, +67s).
INFO  gc.collect() reclaimed 7327 objects in 0.330s                             
Quality evaluation: 4/883
Quality evaluation: 8/883
Quality evaluation: 12/883
Quality evaluation: 16/883
Quality evaluation: 20/883
Quality evaluation: 24/883
Quality evaluation: 28/883
Quality evaluation: 32/883
Quality evaluation: 36/883
Quality evaluation: 40/883
Quality evaluation: 44/883
Quality evaluation: 48/883
Quality evaluation: 52/883
Quality evaluation: 56/883
Quality evaluation: 60/883
Quality evaluation: 64/883
Quality evaluation: 68/883
Quality evaluation: 72/883
Quality evaluation: 76/883
Quality evaluation: 80/883
Quality evaluation: 84/883
Quality evaluation: 88/883
Quality evaluation: 92/883
Quality evaluation: 96/883
Quality evaluation: 100/883
Quality evaluation: 104/883
Quality

Loading weights:   0%|          | 0/794 [00:00<?, ?it/s]

INFO  gc.collect() reclaimed 6735 objects in 0.321s                             
Quality evaluation: 4/883
Quality evaluation: 8/883
Quality evaluation: 12/883
Quality evaluation: 16/883
Quality evaluation: 20/883
Quality evaluation: 24/883
Quality evaluation: 28/883
Quality evaluation: 32/883
Quality evaluation: 36/883
Quality evaluation: 40/883
Quality evaluation: 44/883
Quality evaluation: 48/883
Quality evaluation: 52/883
Quality evaluation: 56/883
Quality evaluation: 60/883
Quality evaluation: 64/883
Quality evaluation: 68/883
Quality evaluation: 72/883
Quality evaluation: 76/883
Quality evaluation: 80/883
Quality evaluation: 84/883
Quality evaluation: 88/883
Quality evaluation: 92/883
Quality evaluation: 96/883
Quality evaluation: 100/883
Quality evaluation: 104/883
Quality evaluation: 108/883
Quality evaluation: 112/883
Quality evaluation: 116/883
Quality evaluation: 120/883
Quality evaluation: 124/883
Quality evaluation: 128/883
Quality evaluation: 132/883
Quality evaluation: 1

Loading weights:   0%|          | 0/794 [00:00<?, ?it/s]

INFO  gc.collect() reclaimed 6735 objects in 0.331s                             
Quality evaluation: 4/883
Quality evaluation: 8/883
Quality evaluation: 12/883
Quality evaluation: 16/883
Quality evaluation: 20/883
Quality evaluation: 24/883
Quality evaluation: 28/883
Quality evaluation: 32/883
Quality evaluation: 36/883
Quality evaluation: 40/883
Quality evaluation: 44/883
Quality evaluation: 48/883
Quality evaluation: 52/883
Quality evaluation: 56/883
Quality evaluation: 60/883
Quality evaluation: 64/883
Quality evaluation: 68/883
Quality evaluation: 72/883
Quality evaluation: 76/883
Quality evaluation: 80/883
Quality evaluation: 84/883
Quality evaluation: 88/883
Quality evaluation: 92/883
Quality evaluation: 96/883
Quality evaluation: 100/883
Quality evaluation: 104/883
Quality evaluation: 108/883
Quality evaluation: 112/883
Quality evaluation: 116/883
Quality evaluation: 120/883
Quality evaluation: 124/883
Quality evaluation: 128/883
Quality evaluation: 132/883
Quality evaluation: 1

In [8]:
%pip -q install autoawq

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.3/74.3 kB 6.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [9]:
for method,root,loader in [('gptq',GPTQ_DIR,load_gptq_service),('awq',AWQ_DIR,load_awq_service)]:
    for group_size in (32,64,128):
        plan_id=f'{method}-int4-w4a16-g{group_size}'; plan_dir=root/plan_id
        service=loader(BASELINE_CONFIG,plan_dir/'quantized_model_manifest.json',MODEL_MANIFEST,plan_dir/f'{method}_comparison_benchmark_service.jsonl')
        report=run_benchmark(service,assets.calibration_prompts[0],benchmark_config)
        report_file=plan_dir/'comparison_benchmark_report.json'
        report_file.write_text(json.dumps(report,ensure_ascii=False,indent=2)+'\n',encoding='utf-8')
        validation=validate_benchmark_report(report_file)
        (plan_dir/'comparison_benchmark_validation.json').write_text(json.dumps(validation.to_dict(),ensure_ascii=False,indent=2)+'\n',encoding='utf-8')
        assert validation.complete
        print(plan_id,'comparison benchmark PASS')
        del service; gc.collect(); torch.cuda.empty_cache()

INFO  Kernel: Auto-selection: adding candidate `MarlinLinear`                   
INFO  Kernel: selected -> `MarlinLinear`.                                       


Loading weights:   0%|          | 0/794 [00:00<?, ?it/s]

INFO  gc.collect() reclaimed 6765 objects in 0.328s                             
Benchmark 1/16: bs1-in128-out16
Benchmark 2/16: bs1-in128-out32
Benchmark 3/16: bs1-in512-out16
Benchmark 4/16: bs1-in512-out32
Benchmark 5/16: bs4-in128-out16
Benchmark 6/16: bs4-in128-out32
Benchmark 7/16: bs4-in512-out16
Benchmark 8/16: bs4-in512-out32
Benchmark 9/16: bs8-in128-out16
Benchmark 10/16: bs8-in128-out32
Benchmark 11/16: bs8-in512-out16
Benchmark 12/16: bs8-in512-out32
Benchmark 13/16: bs16-in128-out16
Benchmark 14/16: bs16-in128-out32
Benchmark 15/16: bs16-in512-out16
Benchmark 16/16: bs16-in512-out32
gptq-int4-w4a16-g32 comparison benchmark PASS
INFO  Kernel: Auto-selection: adding candidate `MarlinLinear`                   
INFO  Kernel: selected -> `MarlinLinear`.                                       


Loading weights:   0%|          | 0/794 [00:00<?, ?it/s]

INFO  gc.collect() reclaimed 6735 objects in 0.321s                             
Benchmark 1/16: bs1-in128-out16
Benchmark 2/16: bs1-in128-out32
Benchmark 3/16: bs1-in512-out16
Benchmark 4/16: bs1-in512-out32
Benchmark 5/16: bs4-in128-out16
Benchmark 6/16: bs4-in128-out32
Benchmark 7/16: bs4-in512-out16
Benchmark 8/16: bs4-in512-out32
Benchmark 9/16: bs8-in128-out16
Benchmark 10/16: bs8-in128-out32
Benchmark 11/16: bs8-in512-out16
Benchmark 12/16: bs8-in512-out32
Benchmark 13/16: bs16-in128-out16
Benchmark 14/16: bs16-in128-out32
Benchmark 15/16: bs16-in512-out16
Benchmark 16/16: bs16-in512-out32
gptq-int4-w4a16-g64 comparison benchmark PASS
INFO  Kernel: Auto-selection: adding candidate `MarlinLinear`                   
INFO  Kernel: selected -> `MarlinLinear`.                                       


...

Benchmark 9/16: bs8-in128-out16
Benchmark 10/16: bs8-in128-out32
Benchmark 11/16: bs8-in512-out16
Benchmark 12/16: bs8-in512-out32
Benchmark 13/16: bs16-in128-out16
Benchmark 14/16: bs16-in128-out32
Benchmark 15/16: bs16-in512-out16
Benchmark 16/16: bs16-in512-out32
awq-int4-w4a16-g32 comparison benchmark PASS


Replacing layers...: 100%|██████████| 24/24 [00:07<00:00,  3.00it/s]


Benchmark 1/16: bs1-in128-out16
Benchmark 2/16: bs1-in128-out32
Benchmark 3/16: bs1-in512-out16
Benchmark 4/16: bs1-in512-out32
Benchmark 5/16: bs4-in128-out16
Benchmark 6/16: bs4-in128-out32
Benchmark 7/16: bs4-in512-out16
Benchmark 8/16: bs4-in512-out32
Benchmark 9/16: bs8-in128-out16
Benchmark 10/16: bs8-in128-out32
Benchmark 11/16: bs8-in512-out16
Benchmark 12/16: bs8-in512-out32
Benchmark 13/16: bs16-in128-out16
Benchmark 14/16: bs16-in128-out32
Benchmark 15/16: bs16-in512-out16
Benchmark 16/16: bs16-in512-out32
awq-int4-w4a16-g64 comparison benchmark PASS


Replacing layers...: 100%|██████████| 24/24 [00:08<00:00,  3.00it/s]


Benchmark 1/16: bs1-in128-out16
Benchmark 2/16: bs1-in128-out32
Benchmark 3/16: bs1-in512-out16
Benchmark 4/16: bs1-in512-out32
Benchmark 5/16: bs4-in128-out16
Benchmark 6/16: bs4-in128-out32
Benchmark 7/16: bs4-in512-out16
Benchmark 8/16: bs4-in512-out32
Benchmark 9/16: bs8-in128-out16
Benchmark 10/16: bs8-in128-out32
Benchmark 11/16: bs8-in512-out16
Benchmark 12/16: bs8-in512-out32
Benchmark 13/16: bs16-in128-out16
Benchmark 14/16: bs16-in128-out32
Benchmark 15/16: bs16-in512-out16
Benchmark 16/16: bs16-in512-out32
awq-int4-w4a16-g128 comparison benchmark PASS
